[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sharukh01621/first-line-to-first-paper/blob/main/colab_drive/solutions/part_07_research_capstone/ch33_reproducible_experiments_ablation/02_guided_todos_SOLUTION.ipynb)

> **From First Line to First Paper** — *Mastering Python, Statistical Data Science, and Applied Deep Learning*  
> **Author:** Mezanur Rahman Sharukh | **Mode:** `ANNOTATED SOLUTION NOTEBOOK`


# Chapter 33 Guided To-Do Practice Labs

**Book Reference:** Chapter 33 (Opening Part VII: Research Capstone), Section 33.5 (Guided To-Do Practice)  
Run **Step 0** to activate `sharukh-tutor`, implement all 6 reproducible-experiment building blocks (multi-library seed control, YAML config validation, bootstrap CI, paired bootstrap test, feature-group ablation, and append-only JSONL ledger), and pass each **function name (without parentheses!)** to `tutor.check(...)`. Use `tutor.hint("ch33_todo_XX", level=1..3)` whenever you want a progressive nudge.

In [ ]:
# @title Step 0: Click Play (▶) to Activate Interactive Hints & Auto-Checker
!pip install -q sharukh-tutor pyyaml
import json
from pathlib import Path
import random
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import torch
import yaml
import sharukh_tutor as tutor

tutor.activate(chapter="ch33", notebook="guided_todos")


---
## To-Do 33.1 (Multi-Library Seed Control): `set_all_seeds(seed=42)`

Implement `set_all_seeds(seed=42)`:
- Call `random.seed(seed)`, `np.random.seed(seed)`, and `torch.manual_seed(seed)`.
- Draw one scalar from each RNG (`random.random()`, `float(np.random.rand())`, `float(torch.rand(1).item())`).
- Return `{"seed": int(seed), "py_draw": round6, "np_draw": round6, "torch_draw": round6}` proving bit-exact reproducibility across repeated calls.

In [ ]:
def set_all_seeds(seed=42):
    """Lock Python, NumPy, and PyTorch RNGs and return 6-dec draws."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    return {
        "seed": int(seed),
        "py_draw": round(float(random.random()), 6),
        "np_draw": round(float(np.random.rand()), 6),
        "torch_draw": round(float(torch.rand(1).item()), 6),
    }


In [ ]:
tutor.check("ch33_todo_01", set_all_seeds)


---
## To-Do 33.2 (Config-Driven YAML Validator): `load_and_validate_config(yaml_path)`

Implement `load_and_validate_config(yaml_path)`:
- Load `yaml_path` via `yaml.safe_load`.
- Verify the parsed object is a `dict` containing all required keys `{"experiment_name", "seeds", "model_type", "cv_splits"}`.
- Verify `seeds` is a non-empty `list` of integers (excluding booleans) and `cv_splits` is an integer `>= 2` (excluding booleans); raise `ValueError` otherwise.
- Return the validated `dict`.

In [ ]:
def load_and_validate_config(yaml_path):
    """Load YAML config and validate required experiment fields."""
    with open(yaml_path, "r", encoding="utf-8") as f:
        cfg = yaml.safe_load(f)
    if not isinstance(cfg, dict):
        raise ValueError("Config YAML must deserialize to a dict.")
    required = {"experiment_name", "seeds", "model_type", "cv_splits"}
    missing = required - set(cfg.keys())
    if missing:
        raise ValueError(f"Missing required keys: {sorted(missing)}")
    seeds = cfg["seeds"]
    valid_seeds = (
        isinstance(seeds, list)
        and len(seeds) > 0
        and all(
            isinstance(s, int) and not isinstance(s, bool)
            for s in seeds
        )
    )
    if not valid_seeds:
        raise ValueError("seeds must be a non-empty list of ints.")
    cv = cfg["cv_splits"]
    if not isinstance(cv, int) or isinstance(cv, bool) or cv < 2:
        raise ValueError("cv_splits must be an integer >= 2.")
    return cfg


In [ ]:
tutor.check("ch33_todo_02", load_and_validate_config)


---
## To-Do 33.3 (Bootstrap Confidence Interval): `bootstrap_metric_ci(y_true, y_score, metric_fn, n_boot=300, alpha=0.05, seed=42)`

Implement `bootstrap_metric_ci(y_true, y_score, metric_fn, n_boot=300, alpha=0.05, seed=42)`:
- Compute `point = metric_fn(y_true, y_score)`.
- Run `n_boot` bootstrap resamples via `np.random.default_rng(seed)`, skipping degenerate single-class resamples (`len(np.unique(y_b)) < 2`).
- Compute percentile CI `[alpha/2, 1 - alpha/2]` and bootstrap SD `se`.
- Return `{"point": round4, "ci_low": round4, "ci_high": round4, "se": round4}`.

In [ ]:
def bootstrap_metric_ci(
    y_true, y_score, metric_fn, n_boot=300, alpha=0.05, seed=42
):
    """Compute percentile bootstrap CI and SE for any scoring fn."""
    yt = np.asarray(y_true)
    ys = np.asarray(y_score)
    point = float(metric_fn(yt, ys))
    rng = np.random.default_rng(seed)
    boots = []
    for _ in range(n_boot):
        idx = rng.integers(0, len(yt), size=len(yt))
        if len(np.unique(yt[idx])) < 2:
            continue
        boots.append(float(metric_fn(yt[idx], ys[idx])))
    lo, hi = np.percentile(
        boots, [100 * alpha / 2, 100 * (1 - alpha / 2)]
    )
    return {
        "point": round(point, 4),
        "ci_low": round(float(lo), 4),
        "ci_high": round(float(hi), 4),
        "se": round(float(np.std(boots, ddof=1)), 4),
    }


In [ ]:
tutor.check("ch33_todo_03", bootstrap_metric_ci)


---
## To-Do 33.4 (Paired Bootstrap Test): `paired_bootstrap_test(y_true, score_a, score_b, metric_fn, n_boot=400, seed=42)`

Implement `paired_bootstrap_test(y_true, score_a, score_b, metric_fn, n_boot=400, seed=42)`:
- Compute `delta_point = metric_fn(y_true, score_b) - metric_fn(y_true, score_a)`.
- On identical bootstrap indices (`rng = np.random.default_rng(seed)`), skipping single-class resamples, compute `delta_i = metric_fn(y_b, sb_b) - metric_fn(y_b, sa_b)`.
- Compute the 95% CI `[2.5, 97.5]` and two-sided bootstrap p-value `2 * min(mean(deltas <= 0), mean(deltas >= 0))` clipped to `[0.0, 1.0]`.
- Return `{"delta_point": round4, "ci_low": round4, "ci_high": round4, "p_value_two_sided": round4}`.

In [ ]:
def paired_bootstrap_test(
    y_true, score_a, score_b, metric_fn, n_boot=400, seed=42
):
    """Run paired bootstrap test comparing score_b minus score_a."""
    yt = np.asarray(y_true)
    sa = np.asarray(score_a)
    sb = np.asarray(score_b)
    delta_point = float(metric_fn(yt, sb) - metric_fn(yt, sa))
    rng = np.random.default_rng(seed)
    deltas = []
    for _ in range(n_boot):
        idx = rng.integers(0, len(yt), size=len(yt))
        if len(np.unique(yt[idx])) < 2:
            continue
        d = float(
            metric_fn(yt[idx], sb[idx]) - metric_fn(yt[idx], sa[idx])
        )
        deltas.append(d)
    arr = np.asarray(deltas)
    lo, hi = np.percentile(arr, [2.5, 97.5])
    p_raw = 2.0 * min(float(np.mean(arr <= 0)), float(np.mean(arr >= 0)))
    p_val = float(np.clip(p_raw, 0.0, 1.0))
    return {
        "delta_point": round(delta_point, 4),
        "ci_low": round(float(lo), 4),
        "ci_high": round(float(hi), 4),
        "p_value_two_sided": round(p_val, 4),
    }


In [ ]:
tutor.check("ch33_todo_04", paired_bootstrap_test)


---
## To-Do 33.5 (Feature-Group Ablation): `run_feature_ablation(X_df, y, feature_groups, model_factory, cv_splits=5, seed=42)`

Implement `run_feature_ablation(X_df, y, feature_groups, model_factory, cv_splits=5, seed=42)`:
- Evaluate `"full_model"` and then `"drop_<group_name>"` for each group in `feature_groups` without mutating `X_df`.
- Use `Pipeline([("scaler", StandardScaler()), ("model", model_factory())])` + `StratifiedKFold(cv_splits, shuffle=True, random_state=seed)` scoring `"roc_auc"`.
- Return a `pd.DataFrame` with columns `["ablation", "mean_auc", "std_auc", "delta_vs_full"]` rounded to 4 decimals (`delta_vs_full = mean_auc - full_auc`, so negative means dropping that group hurt!).

In [ ]:
def run_feature_ablation(
    X_df, y, feature_groups, model_factory, cv_splits=5, seed=42
):
    """Run leak-free leave-one-group-out feature ablation via CV."""
    skf = StratifiedKFold(
        n_splits=cv_splits, shuffle=True, random_state=seed
    )
    pipe_full = Pipeline([
        ("scaler", StandardScaler()),
        ("model", model_factory()),
    ])
    full_sc = cross_val_score(
        pipe_full, X_df, y, cv=skf, scoring="roc_auc"
    )
    full_auc = float(full_sc.mean())
    rows = [{
        "ablation": "full_model",
        "mean_auc": round(full_auc, 4),
        "std_auc": round(float(full_sc.std()), 4),
        "delta_vs_full": 0.0,
    }]
    for gname, cols in feature_groups.items():
        pipe_g = Pipeline([
            ("scaler", StandardScaler()),
            ("model", model_factory()),
        ])
        sc = cross_val_score(
            pipe_g, X_df.drop(columns=cols), y,
            cv=skf, scoring="roc_auc",
        )
        m = float(sc.mean())
        rows.append({
            "ablation": f"drop_{gname}",
            "mean_auc": round(m, 4),
            "std_auc": round(float(sc.std()), 4),
            "delta_vs_full": round(m - full_auc, 4),
        })
    return pd.DataFrame(rows)


In [ ]:
tutor.check("ch33_todo_05", run_feature_ablation)


---
## To-Do 33.6 (Experiment Ledger): `append_experiment_ledger(ledger_path, run_record)`

Implement `append_experiment_ledger(ledger_path, run_record)`:
- Append one UTF-8 JSON line (`json.dumps(run_record, sort_keys=True) + "\n"`) to `ledger_path`.
- Read back `ledger_path` and return the total number of non-empty lines as an `int`.

In [ ]:
def append_experiment_ledger(ledger_path, run_record):
    """Append one UTF-8 JSONL record and return total line count."""
    path = Path(ledger_path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(run_record, sort_keys=True) + "\n")
    with open(path, "r", encoding="utf-8") as f:
        return int(sum(1 for line in f if line.strip()))


In [ ]:
tutor.check("ch33_todo_06", append_experiment_ledger)


---
## Annotated Debugging Post-Mortem (Solution Notebook Only)

- **To-Do 33.2 (Python's `bool` Is a Subclass of `int`!):** Checking `isinstance(x, int)` alone silently accepts `True` and `False` inside YAML configs (`seeds: [true]` or `cv_splits: true`). Always combine `isinstance(x, int) and not isinstance(x, bool)` when validating integer config parameters.
- **To-Do 33.3 & 33.4 (Degenerate Resamples & Clipped Two-Sided p-Values):** On rare-event medical cohorts, a bootstrap draw occasionally contains only class `0`, crashing `roc_auc_score` unless you guard with `if len(np.unique(y_b)) < 2: continue`. Likewise, when two models tie identically (`deltas == 0`), `2 * min(mean(deltas <= 0), mean(deltas >= 0))` equals `2.0` unless clipped cleanly to `[0.0, 1.0]`.
- **To-Do 33.6 (Append Mode `'a'` vs. Overwrite `'w'`):** Opening a JSONL ledger in `'w'` mode destroys every earlier benchmark run on the second call. Always use `'a'` mode with `sort_keys=True` so every run is immutable and diff-friendly.

In [ ]:
tutor.summary()
